# Summary Visualizations

Charts compiled from the results already computed in
notebooks 02-07. These summarize across hypotheses rather than rerunning
any modeling, so the coefficients/CIs below are transcribed from the source
notebooks' verified output.

In [1]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

blue = '#2a78d6'
orange = '#eb6834'
critical = '#d03b3b'
muted = '#898781'
grid = '#e1e0d9'
ink = '#0b0b0b'
secondary_ink = '#52514e'
surface = '#fcfcfb'

## Chart 1: Primary coefficient across all four hypotheses

In [2]:
hypotheses = [
    dict(name='H1: FT3/FT4 ratio<br>-> triglycerides', coef=20.334, lo=13.190, hi=27.479, sig=True, xlabel='mg/dL triglycerides per unit ratio'),
    dict(name='H2: free T3<br>-> metabolic syndrome', coef=0.379, lo=0.196, hi=0.562, sig=True, xlabel='standardized log-odds'),
    dict(name='H3: free T3<br>-> log(CRP)', coef=0.227, lo=0.083, hi=0.370, sig=True, xlabel='log-CRP units per unit free T3'),
    dict(name='H4: free T3<br>-> total cholesterol', coef=4.525, lo=-0.798, hi=9.848, sig=False, xlabel='mg/dL cholesterol per unit free T3'),
]

fig1 = make_subplots(rows=1, cols=4, subplot_titles=[h['name'] for h in hypotheses])

for i, h in enumerate(hypotheses):
    color = critical if h['sig'] else muted
    fig1.add_trace(go.Scatter(
        x=[h['coef']], y=[0],
        error_x=dict(type='data', symmetric=False, array=[h['hi']-h['coef']], arrayminus=[h['coef']-h['lo']], color=color, thickness=2, width=6),
        mode='markers', marker=dict(color=color, size=12),
        showlegend=False,
        hovertemplate=f"coef={h['coef']:.3f}<br>95% CI [{h['lo']:.3f}, {h['hi']:.3f}]<extra></extra>",
    ), row=1, col=i+1)
    fig1.add_vline(x=0, line_dash='dash', line_color=grid, row=1, col=i+1)
    fig1.update_xaxes(title_text=h['xlabel'], title_font_size=10, row=1, col=i+1, gridcolor=grid, zeroline=False, linecolor=muted)
    fig1.update_yaxes(showticklabels=False, row=1, col=i+1, gridcolor=grid, zeroline=False)

fig1.update_layout(
    title='Primary coefficient across all four hypotheses (red = survives multiple-comparison correction)',
    plot_bgcolor=surface, paper_bgcolor=surface,
    font=dict(color=ink, family='system-ui, -apple-system, "Segoe UI", sans-serif'),
    height=400, width=1100,
    margin=dict(t=100),
)
for ann in fig1['layout']['annotations']:
    ann['font'] = dict(color=secondary_ink, size=12)

fig1.show()

## Chart 2: Raw vs. Corrected p-values

In [3]:
aqua = '#1baf7a'

hyps = ['H1\ntriglycerides', 'H2\nT3 vs TSH', 'H3\nCRP', 'H4\ncholesterol']
from statsmodels.stats.multitest import multipletests

# Raw primary-test p-values from notebook 06; corrections recomputed here
# rather than transcribed.
raw = [7.36e-07, 0.000133, 0.00311, 0.0938]
fdr = multipletests(raw, method='fdr_bh')[1]
bonf = multipletests(raw, method='bonferroni')[1]

fig2 = go.Figure()
fig2.add_trace(go.Bar(name='Raw p-value', x=hyps, y=raw, marker_color=blue))
fig2.add_trace(go.Bar(name='FDR-adjusted', x=hyps, y=fdr, marker_color=orange))
fig2.add_trace(go.Bar(name='Bonferroni-adjusted', x=hyps, y=bonf, marker_color=aqua))

fig2.add_hline(y=0.05, line_dash='dash', line_color=critical, annotation_text='alpha = 0.05', annotation_position='top left', annotation_font_color=critical)

fig2.update_yaxes(type='log', exponentformat='power', title='p-value (log scale)', gridcolor=grid, zeroline=False, linecolor=muted)
fig2.update_xaxes(title='Hypothesis', gridcolor=grid, zeroline=False, linecolor=muted)

fig2.update_layout(
    title='Raw vs. Corrected p-values across all four hypotheses',
    barmode='group',
    plot_bgcolor=surface, paper_bgcolor=surface,
    font=dict(color=ink, family='system-ui, -apple-system, "Segoe UI", sans-serif'),
    legend=dict(font=dict(color=secondary_ink)),
    height=450, width=800,
)

fig2.show()

## Chart 3: H4 Primary vs. Sensitivity

In [4]:
rows = [
    dict(label='Primary (excludes lipid-lowering meds, n=6,283)', coef=4.525, lo=-0.798, hi=9.848),
    dict(label='Sensitivity (includes + adjusts, n=7,696)', coef=6.042, lo=0.976, hi=11.109),
]

fig3 = go.Figure()
for r in rows:
    fig3.add_trace(go.Scatter(
        x=[r['coef']], y=[r['label']],
        error_x=dict(type='data', symmetric=False, array=[r['hi']-r['coef']], arrayminus=[r['coef']-r['lo']], color=muted, thickness=2, width=8),
        mode='markers', marker=dict(color=muted, size=14),
        showlegend=False,
        hovertemplate=f"coef={r['coef']:.3f}<br>95% CI [{r['lo']:.3f}, {r['hi']:.3f}]<extra></extra>",
    ))

fig3.add_vline(x=0, line_dash='dash', line_color='#c3c2b7')

fig3.update_xaxes(title='mg/dL total cholesterol per unit free T3', gridcolor=grid, zeroline=False, linecolor=muted)
fig3.update_yaxes(gridcolor=grid, zeroline=False, automargin=True)

fig3.update_layout(
    title='H4: free T3 -> total cholesterol, primary model null, sensitivity model positive',
    plot_bgcolor=surface, paper_bgcolor=surface,
    font=dict(color=ink, family='system-ui, -apple-system, "Segoe UI", sans-serif'),
    height=300, width=700,
    margin=dict(l=280),
)

fig3.show()

## Chart 4: H5 Exploratory Markers

In [5]:
tests = [
    dict(name='Pulse -><br>triglycerides', coef=1.281, lo=0.471, hi=2.091, marker='pulse', xlabel='mg/dL per bpm'),
    dict(name='Pulse -><br>MetS', coef=0.0257, lo=0.0127, hi=0.0387, marker='pulse', xlabel='log-odds per bpm'),
    dict(name='Pulse -><br>log(CRP)', coef=0.0176, lo=0.0146, hi=0.0207, marker='pulse', xlabel='log-CRP units per bpm'),
    dict(name='Cholesterol -><br>MetS', coef=0.0066, lo=0.0028, hi=0.0104, marker='cholesterol', xlabel='log-odds per mg/dL'),
    dict(name='Cholesterol -><br>log(CRP)', coef=0.0018, lo=0.0008, hi=0.0028, marker='cholesterol', xlabel='log-CRP units per mg/dL'),
]

fig4 = make_subplots(rows=1, cols=5, subplot_titles=[t['name'] for t in tests])

for i, t in enumerate(tests):
    color = blue if t['marker'] == 'pulse' else orange
    fig4.add_trace(go.Scatter(
        x=[t['coef']], y=[0],
        error_x=dict(type='data', symmetric=False, array=[t['hi']-t['coef']], arrayminus=[t['coef']-t['lo']], color=color, thickness=2, width=6),
        mode='markers', marker=dict(color=color, size=12),
        showlegend=False,
        hovertemplate=f"coef={t['coef']:.4f}<br>95% CI [{t['lo']:.4f}, {t['hi']:.4f}]<extra></extra>",
    ), row=1, col=i+1)
    fig4.add_vline(x=0, line_dash='dash', line_color=grid, row=1, col=i+1)
    fig4.update_xaxes(title_text=t['xlabel'], title_font_size=9, row=1, col=i+1, gridcolor=grid, zeroline=False, linecolor=muted)
    fig4.update_yaxes(showticklabels=False, row=1, col=i+1, gridcolor=grid, zeroline=False)

fig4.add_trace(go.Scatter(x=[None], y=[None], mode='markers', marker=dict(color=blue, size=12), name='Pulse rate (reinforces disconfirmation)'))
fig4.add_trace(go.Scatter(x=[None], y=[None], mode='markers', marker=dict(color=orange, size=12), name='Cholesterol (confounded, see caveat)'))

fig4.update_layout(
    title='H5 Exploratory: all five tests significant, but the two markers mean different things',
    plot_bgcolor=surface, paper_bgcolor=surface,
    font=dict(color=ink, family='system-ui, -apple-system, "Segoe UI", sans-serif'),
    legend=dict(font=dict(color=secondary_ink), orientation='h', yanchor='bottom', y=-0.35, xanchor='center', x=0.5),
    height=430, width=1150,
    margin=dict(t=100, b=120),
)
for ann in fig4['layout']['annotations']:
    ann['font'] = dict(color=secondary_ink, size=11)

fig4.show()

In [6]:
# Export static images for the README / future website
fig1.write_image('../docs/figures/summary_all_hypotheses.png')
fig2.write_image('../docs/figures/multiplicity_correction.png')
fig3.write_image('../docs/figures/h4_primary_vs_sensitivity.png')
fig4.write_image('../docs/figures/h5_exploratory_markers.png')